## Script for histograms creation

In [39]:
%reload_ext autoreload
%autoreload 2

import glob
import json
import time
import os
import hashlib


import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast, log_analysis_run, build_cut_expression
from utils.hist_utils import (
    split_variable_group,
    fill_1d_histograms,
    fill_2d_histograms,
    draw_and_save_1d_histograms,
    draw_and_save_2d_histograms,
)

ROOT.EnableImplicitMT(90)
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gSystem.Load("/workspace/KLOE/Subanalysis/CPFit/libCPFit.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/FortranAnalysis/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Subanalysis/CPFit/inc")

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/FortranAnalysis/inc/fort_common.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/const.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/HypothesisCodes.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/SelectionCodes.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/charged_mom.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/kloe_class.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Subanalysis/CPFit/inc/cpfit.hpp"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

print(channName)

{0: 'Data', 1: 'Signal', 2: 'Regeneration', 3: 'Omega', 4: '3pi0', 5: 'Semileptonic', 6: 'Other', 7: 'pi+pi-pi+pi-', 8: 'MC sum'}


In [66]:
base_path = "/workspace/root_notebooks/results"

################ SETTINGS ##################################
analysis_dir = "signal_separated_kin_fits"
img_dir = "img"
date = "2026-10-22"
root_file_name = "separated_kin_fits_1D_variables.root"
eff_graph_file_name = "efficiency_purity_graphs_tag_000_done_signal_selection.root"
mode = "full"
nocut_hash = "09acde96"
log_flag = False
draw_data_mc_sum = False
############################################################

hist_file_path = f"{base_path}/{analysis_dir}/{date}/{root_file_name}"
eff_graph_file_path = f"{base_path}/{analysis_dir}/{date}/{eff_graph_file_name}"

#################### ANALYSIS REGISTRY #####################
import pandas as pd
analysis_registry = f"{base_path}/{analysis_dir}/{date}/analysis_registry.csv"
df = pd.read_csv(analysis_registry)

hash_available = df["hash"]

## Making histograms

In [67]:
# Generate overlapped histograms for all channels
variables_1D = ["Knerec_5", 
                "KnerecSix_5", 
                "Kchrec_5",
                "Chi2SignalKinFit",
                "Chi2NeutralKinFit",
                "Chi2PMKinFit",
                "MassPi01Fit",
                "MassPi02Fit",
                "MassPi0NeutralFit1",
                "MassPi0NeutralFit2",
                "MassPi0PMFit1",
                "MassPi0PMFit2",
                "MassPi01",
                "MassPi02",
                "Delta_t_fit",
                "Delta_t_tri_fit",
                "Delta_t_rec",
                "nclu",
                "bestErrorSixGamma",
                "neu_clu_num"
                ]
variables_1D_titles = ["m^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]", 
                       "m^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]", 
                       "m^{inv}_{+-} - m_{K^{0}} [MeV/c^{2}]",
                       "#chi^{2}_{+-00} [-]",
                       "#chi^{2}_{00} [-]",
                       "#chi^{2}_{+-} [-]",
                       "m^{inv,fit,+-00}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-00}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,00}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,00}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,rec}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,rec}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "#Deltat_{fit} [#tau_{S}]",
                       "#Deltat_{tri} [#tau_{S}]",
                       "#Deltat_{rec} [#tau_{S}]",
                       "Total number of clusters",
                       "#Delta_{6} [cm^{2}]",
                       "Number of neutral clusters"
                       ]

unit_1D = ["MeV/c^{2}", 
            "MeV/c^{2}", 
            "MeV/c^{2}",
            "",
            "",
            "",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "#tau_{S}",
            "#tau_{S}",
            "#tau_{S}",
            "",
            "cm^{2}",
            ""
            ]

range_X_1D = [(-400, 200), 
              (-400, 200), 
              (-200, 200),
              (0, 500),
              (0, 500),
              (0, 100),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-300, 300),
              (-300, 300),
              (-300, 300),
              (0, 12),
              (0, 10000),
              (0, 12)
              ]

TRUTH_ONLY_VARS_1D = {}
TRUTH_MCTRUTH_ID_1D = 1

In [68]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)
ROOT.TH1.AddDirectory(False)

#################### OBJECT TO FIND ################################
histograms = {}

for hash in hash_available:
    if hash not in [key.GetName() for key in hist_file.GetListOfKeys()]:
        continue
    
    histograms[hash] = {}

    for var in variables_1D:
      dir_obj = hist_file.Get(f"{hash}/{var}")
      if var not in histograms[hash]:
          histograms[hash][var] = {}

      for chann in channName.values():
        hist_name = f"{var}_{chann}_{mode}"

        if hist_name not in [key.GetName() for key in dir_obj.GetListOfKeys()]:
            continue
        histograms[hash][var][chann] = dir_obj.Get(hist_name).Clone()

hist_file.Close()


AttributeError: 'TObject' object has no attribute 'GetListOfKeys'

In [65]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{date}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

################## DRAW USING THSTACK ###########################
canvases = []
legends = []
stacks = []

for hash in histograms:    
    path_hash = f"{img_dir_path}/{hash}"
    os.makedirs(path_hash, exist_ok=True)
    for i, var in enumerate(histograms[hash]):
        path_var = f"{path_hash}/{var}"
        os.makedirs(path_var, exist_ok=True)
        
        legend = ROOT.TLegend(0.15, 0.75, 0.84875, 0.9)
        legend.SetTextFont(62)
        legend.SetTextSize(0.025)
        legend.SetMargin(0.1)
        legend.SetNColumns(2)
        
        # Dodajemy hash do nazwy, aby uniknąć konfliktów w pamięci ROOTa przy wielu folderach
        stack = ROOT.THStack(f"{var}_stack_{hash}", "")
        
        for flag in histograms[hash][var]:
            hist = histograms[hash][var][flag]

            if not draw_data_mc_sum and (flag == "Data" or flag == "MC sum"):
                continue

            if hist:
                hist.SetLineColor(channColor[flag])

                if log_flag:
                    hist.SetMinimum(0.1)
                
                if flag != "Data":
                    # Opcję rysowania przekazujemy bezpośrednio w metodzie Add dla THStack
                    stack.Add(hist, "hist")
                    legend.AddEntry(hist, channTitle[flag], "l")
                else:
                    hist.SetMarkerStyle(20)
                    hist.SetMarkerColor(channColor[flag])
                    stack.Add(hist, "pe1")
                    legend.AddEntry(hist, channTitle[flag], "pe1")
                            
        canvas = ROOT.TCanvas(f"{var}_canvas_{hash}", f"{var} Canvas", 600, 600)
        
        stack.Draw("NOSTACK")
        stack.SetMaximum(stack.GetMaximum() * 1.0)
        
        if stack.GetHistogram():
            resolution = stack.GetHistogram().GetXaxis().GetBinWidth(1)
            
            stack.GetHistogram().GetXaxis().SetTitle(variables_1D_titles[i])
            stack.GetHistogram().GetYaxis().SetTitle(f"Counts / {resolution:.2f} {unit_1D[i]}")

        ####### Set log y ########
        if log_flag:
          stack.SetMinimum(0.1)
          stack.SetMaximum(stack.GetMaximum() * 50)
        else:
          stack.SetMinimum(0)
                
        canvas.SetLogy(log_flag)
        ##########################
            
        legend.Draw()

        canvas.Draw()
        canvas.SaveAs(f"{path_var}/{var}_{mode}_logy_{log_flag}.pdf")
        canvas.Close()

Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/Knerec_5/Knerec_5_full_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/KnerecSix_5/KnerecSix_5_full_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/Kchrec_5/Kchrec_5_full_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/Chi2SignalKinFit/Chi2SignalKinFit_full_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/Chi2NeutralKinFit/Chi2NeutralKinFit_full_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/Chi2PMKinFit/

## Check of purity in the function of $\Delta t$

In [6]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{date}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)



delta_t_vars = ["Delta_t_fit",
                "Delta_t_tri_fit",
                "Delta_t_rec"]

delta_t_titles = ["#Deltat_{fit} [#tau_{S}]",
                  "#Deltat_{tri} [#tau_{S}]",
                  "#Deltat_{rec} [#tau_{S}]"]

signal = "3pi0"
sum = "MC sum"

delta_t_signal = {}
delta_t_sum = {}

for hash in histograms:
    delta_t_signal[hash] = []
    delta_t_sum[hash] = []
    for var in delta_t_vars:
        if var not in histograms[hash]:
            continue

        delta_t_signal[hash].append(histograms[hash][var][signal])
        delta_t_sum[hash].append(histograms[hash][var][sum])


######################### DIVIDE #############################
delta_t_divided = {}

for hash in delta_t_signal:
    delta_t_divided[hash] = []
    for signal_hist, sum_hist in zip(delta_t_signal[hash], delta_t_sum[hash]):
        hist_temp = signal_hist.Clone()
        hist_temp.Divide(sum_hist)
        delta_t_divided[hash].append(hist_temp)


######################## PLOT ###########################
canvases = []

ROOT.gStyle.SetOptStat(0)

for hash in delta_t_divided:
    path_hash = f"{img_dir_path}/{hash}"
    os.makedirs(path_hash, exist_ok=True)
    for hist, title in zip(delta_t_divided[hash], delta_t_titles):
        c = ROOT.TCanvas("")
        hist.SetMarkerStyle(ROOT.kFullCircle)
        hist.SetLineColor(ROOT.kBlack)
        hist.GetYaxis().SetRangeUser(0, 1.1)
        hist.GetYaxis().SetTitle("Purity")
        hist.Draw("PE1")
        c.Draw()
        c.SaveAs(f"{path_hash}/purity_{title.replace(' ', '_')}_per_channel.pdf")
        canvases.append(c)

        

Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/09acde96/purity_#Deltat_{fit}_[#tau_{S}]_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/09acde96/purity_#Deltat_{tri}_[#tau_{S}]_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/09acde96/purity_#Deltat_{rec}_[#tau_{S}]_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/6d56cd00/purity_#Deltat_{fit}_[#tau_{S}]_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/6d56cd00/purity_#Deltat_{tri}_[#tau_{S}]_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/

## Check of efficiency in the function of $\Delta t$

In [7]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{date}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)



delta_t_vars = ["Delta_t_fit",
                "Delta_t_tri_fit",
                "Delta_t_rec"]

delta_t_titles = ["#Deltat_{fit} [#tau_{S}]",
                  "#Deltat_{tri} [#tau_{S}]",
                  "#Deltat_{rec} [#tau_{S}]"]

signal = "3pi0"
sum = "3pi0"

delta_t_signal = {}
delta_t_sum = {}

for hash in histograms:
    delta_t_signal[hash] = []
    delta_t_sum[hash] = []
    for var in delta_t_vars:
        if var not in histograms[hash]:
            continue

        delta_t_signal[hash].append(histograms[hash][var][signal])
        delta_t_sum[hash].append(histograms[nocut_hash][var][sum])


######################### DIVIDE #############################
delta_t_divided_signal = {}

for hash in delta_t_signal:
    delta_t_divided_signal[hash] = []
    for signal_hist, sum_hist in zip(delta_t_signal[hash], delta_t_sum[hash]):
        hist_temp = signal_hist.Clone()
        hist_temp.Divide(sum_hist)
        delta_t_divided_signal[hash].append(hist_temp)


######################## PLOT ###########################
canvases = []

ROOT.gStyle.SetOptStat(0)

for hash in delta_t_divided_signal:
    path_hash = f"{img_dir_path}/{hash}"
    os.makedirs(path_hash, exist_ok=True)
    for hist, title in zip(delta_t_divided_signal[hash], delta_t_titles):
        c = ROOT.TCanvas("")
        hist.SetMarkerStyle(ROOT.kFullCircle)
        hist.SetLineColor(ROOT.kBlack)
        hist.GetYaxis().SetRangeUser(0, 1.1)
        hist.GetYaxis().SetTitle("Efficiency")
        hist.Draw("PE1")
        c.Draw()
        c.SaveAs(f"{path_hash}/efficiency_{title.replace(' ', '_')}_per_channel.pdf")
        canvases.append(c)

Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/09acde96/efficiency_#Deltat_{fit}_[#tau_{S}]_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/09acde96/efficiency_#Deltat_{tri}_[#tau_{S}]_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/09acde96/efficiency_#Deltat_{rec}_[#tau_{S}]_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/6d56cd00/efficiency_#Deltat_{fit}_[#tau_{S}]_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/6d56cd00/efficiency_#Deltat_{tri}_[#tau_{S}]_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sa

In [8]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{date}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

delta_t_vars = ["Delta_t_fit",
                "Delta_t_tri_fit",
                "Delta_t_rec"]

delta_t_titles = ["#Deltat_{fit} [#tau_{S}]",
                  "#Deltat_{tri} [#tau_{S}]",
                  "#Deltat_{rec} [#tau_{S}]"]

signal = "Data"
signal_MC = "3pi0"
sum = "MC sum"

delta_t_signal = {}
delta_t_sum = {}

for hash in histograms:
    delta_t_signal[hash] = []
    delta_t_sum[hash] = []
    for var in delta_t_vars:
        if var not in histograms[hash]:
            continue

        hist_data_tmp = histograms[hash][var][signal].Clone(f"{hash}_{var}_{signal}")
        hist_data_nocut_tmp = histograms[nocut_hash][var][signal].Clone(f"{nocut_hash}_{var}_{signal}")

        for key in histograms[hash][var]:
            if key == signal or key == signal_MC or key == sum:
              continue

            hist_data_tmp.Add(histograms[hash][var][key])
            hist_data_nocut_tmp.Add(histograms[nocut_hash][var][key])
              
        delta_t_signal[hash].append(hist_data_tmp)
        delta_t_sum[hash].append(hist_data_nocut_tmp)


######################### DIVIDE #############################
delta_t_divided_data = {}

for hash in delta_t_signal:
    delta_t_divided_data[hash] = []
    for signal_hist, sum_hist in zip(delta_t_signal[hash], delta_t_sum[hash]):
        hist_temp = signal_hist.Clone()
        hist_temp.Sumw2()
        hist_temp.Divide(sum_hist)
        delta_t_divided_data[hash].append(hist_temp)


######################## PLOT ###########################
canvases = []

ROOT.gStyle.SetOptStat(0)

for hash in delta_t_divided_data:
    path_hash = f"{img_dir_path}/{hash}"
    os.makedirs(path_hash, exist_ok=True)
    for hist, title in zip(delta_t_divided_data[hash], delta_t_titles):
        c = ROOT.TCanvas("")
        hist.SetMarkerStyle(ROOT.kFullCircle)
        hist.SetLineColor(ROOT.kBlack)
        hist.GetYaxis().SetRangeUser(0, 1.1)
        hist.GetYaxis().SetTitle("Efficiency")
        hist.Draw("PE1")
        c.Draw()
        c.SaveAs(f"{path_hash}/efficiency_{title.replace(' ', '_')}_data.pdf")
        canvases.append(c)

Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/res

## Calc correction factor for $K_{S}\to\pi^+\pi^-$

In [9]:
corr_factor = {}

if delta_t_divided_data and delta_t_divided_signal:
    for hash in delta_t_divided_data:
        corr_factor[hash] = []
        for hist_data, hist_tag in zip(delta_t_divided_data[hash], delta_t_divided_signal[hash]):
            hist_temp = hist_data.Clone()
            hist_temp.Sumw2()
            hist_temp.Divide(hist_tag)
            corr_factor[hash].append(hist_temp)

    for hash in corr_factor:
        path_hash = f"{img_dir_path}/{hash}"
        os.makedirs(path_hash, exist_ok=True)
        for hist, title in zip(corr_factor[hash], delta_t_titles):
            c = ROOT.TCanvas("")
            hist.SetMarkerStyle(ROOT.kFullCircle)
            hist.SetLineColor(ROOT.kBlack)
            hist.GetYaxis().SetRangeUser(0, 1.1)
            hist.GetYaxis().SetTitle("Correction Factor")
            hist.Draw("PE1")
            c.Draw()
            c.SaveAs(f"{path_hash}/corr_factor_{title.replace(' ', '_')}.pdf")
            canvases.append(c)

Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/res

## Making efficiency - purity plots

In [7]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(eff_graph_file_path)
#################### OBJECT TO FIND ################################
graphs = {}
var_dict = {}

for hash in hash_available:
    if hash not in hist_file.GetListOfKeys():
        continue

    graphs[hash] = {}
    obj = hist_file.Get(hash)

    if obj:
      var_list = [key.GetName() for key in obj.GetListOfKeys()]
      var_dict[hash] = var_list
      for var in var_list:
          graphs[hash][var] = {}
          dir_obj = obj.Get(var)
          for type in ["efficiency", "purity"]:
            graphs[hash][var][type] = {}
            for flag in flags:
                graph_name = f"{type}_{flag}"
                graphs[hash][var][type][flag] = dir_obj.Get(graph_name).Clone()

hist_file.Close()

print("Graphs loaded:", graphs)

Graphs loaded: {'519c0397': {'bestErrorSixGamma': {'efficiency': {'has000': <cppyy.gbl.TGraph object at 0x56231c9f13a0>, 'no000': <cppyy.gbl.TGraph object at 0x56231ca88160>, 'Data': <cppyy.gbl.TGraph object at 0x56231cae10c0>, 'MC sum': <cppyy.gbl.TGraph object at 0x56231caafd80>}, 'purity': {'has000': <cppyy.gbl.TGraph object at 0x56231cab6cf0>, 'no000': <cppyy.gbl.TGraph object at 0x56231ca8df20>, 'Data': <cppyy.gbl.TGraph object at 0x56231caf8570>, 'MC sum': <cppyy.gbl.TGraph object at 0x56231caecba0>}}}, '66563976': {'Chi2NeutralKinFit': {'efficiency': {'has000': <cppyy.gbl.TGraph object at 0x56231ca1ee90>, 'no000': <cppyy.gbl.TGraph object at 0x56231ca98780>, 'Data': <cppyy.gbl.TGraph object at 0x56231ca6b1a0>, 'MC sum': <cppyy.gbl.TGraph object at 0x56231c8beba0>}, 'purity': {'has000': <cppyy.gbl.TGraph object at 0x56231c482870>, 'no000': <cppyy.gbl.TGraph object at 0x56231c9f4020>, 'Data': <cppyy.gbl.TGraph object at 0x56231cab8680>, 'MC sum': <cppyy.gbl.TGraph object at 0x5623

In [10]:
%jsroot off

img_dir_path = f"{base_path}/{analysis_dir}/{date}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

var_titles_graph = dict(zip(variables_1D, variables_1D_titles))

canvases = []
multigraphs = []
legends = []

################## DRAW USING THSTACK ###########################
for hash in hash_available:
  if hash not in graphs:
      continue
  path_hash = f"{img_dir_path}/{hash}/optimization"
  os.makedirs(path_hash, exist_ok=True)

  for i, var in enumerate(graphs[hash]):

    legend = ROOT.TLegend(0.5, 0.7, 0.9, 0.9)
    legend.SetTextFont(62)
    legend.SetTextSize(0.025)
    legend.SetMargin(0.1)

    mg = ROOT.TMultiGraph(f"mg_{hash}_{var}", f";{var_titles_graph[var]};Value")

    # Dodano opcję "l" (Line), żeby TMultiGraph wiedział, jak narysować TGraph
    mg.Add(graphs[hash][var]["efficiency"]["has000"], "l")
    mg.Add(graphs[hash][var]["purity"]["has000"], "l")
    mg.Add(graphs[hash][var]["efficiency"]["no000"], "l")

    legend.AddEntry(graphs[hash][var]["efficiency"]["has000"], "Efficiency for K_{L} #rightarrow 3#pi^{0}", "l")
    legend.AddEntry(graphs[hash][var]["purity"]["has000"], "Purity for K_{L} #rightarrow 3#pi^{0}", "l")
    legend.AddEntry(graphs[hash][var]["efficiency"]["no000"], "Efficiency without K_{L} #rightarrow 3#pi^{0}", "l")

    c = ROOT.TCanvas(f"c_graph_{var}_{hash}")
    canvases.append(c)
    multigraphs.append(mg)
    legends.append(legend)
    mg.Draw("AL")
    legend.Draw()
    
    # Wymagane odświeżenie przed zapisem, żeby ROOT poprawnie wyrenderował osie i linie
    c.Modified()
    c.Update()

    c.Draw()
    c.SaveAs(f"{path_hash}/{var}.pdf")

SegmentationViolation: segfault in C++; program state was reset

 *** Break *** segmentation violation
 Generating stack trace...
 0x00007feda3d99552 in TPad::PaintModified() + 0x232 from /opt/root/lib/libGpad.so
 0x00007feda3d5cc84 in TCanvas::Update() at TCanvas.cxx:? from /opt/root/lib/libGpad.so
 0x00007fedc71daa4f in <unknown> from /opt/root/lib/libcppyy_backend.so
 0x00007feda99cfe45 in <unknown> from /opt/root/lib/libcppyy.so
 0x00007feda99b9166 in <unknown> from /opt/root/lib/libcppyy.so
 0x00007feda99bc350 in CPyCppyy::CPPMethod::ExecuteProtected(void*, long, CPyCppyy::CallContext*) + 0x170 from /opt/root/lib/libcppyy.so
 0x00007feda99ba8f7 in CPyCppyy::CPPMethod::Execute(void*, long, CPyCppyy::CallContext*) + 0x27 from /opt/root/lib/libcppyy.so
 0x00007feda99bb3b2 in CPyCppyy::CPPMethod::Call(CPyCppyy::CPPInstance*&, _object* const*, unsigned long, _object*, CPyCppyy::CallContext*) + 0x102 from /opt/root/lib/libcppyy.so
 0x00007feda99c16b1 in <unknown> from /opt/root/lib/libcppyy.so
 0x00007fedebb5f57b in _PyEval_EvalFrameDefault + 0x75b f

Error in callback <bound method CaptureDrawnPrimitives._post_execute of <JupyROOT.helpers.utils.CaptureDrawnPrimitives object at 0x7fede99397c0>> (for post_execute), with arguments args (),kwargs {}:


 Generating stack trace...
 0x00007feda3d9902a in TPad::Paint(char const*) + 0x14a from /opt/root/lib/libGpad.so
 0x00007feda3d8f040 in TPad::Print(char const*, char const*) + 0x1e40 from /opt/root/lib/libGpad.so
 0x00007feda3d859d4 in TPad::SaveAs(char const*, char const*) const + 0x664 from /opt/root/lib/libGpad.so
 0x00007fedc71dae29 in <unknown> from /opt/root/lib/libcppyy_backend.so
 0x00007feda99cfe45 in <unknown> from /opt/root/lib/libcppyy.so
 0x00007feda99b9166 in <unknown> from /opt/root/lib/libcppyy.so
 0x00007feda99bc350 in CPyCppyy::CPPMethod::ExecuteProtected(void*, long, CPyCppyy::CallContext*) + 0x170 from /opt/root/lib/libcppyy.so
 0x00007feda99ba8f7 in CPyCppyy::CPPMethod::Execute(void*, long, CPyCppyy::CallContext*) + 0x27 from /opt/root/lib/libcppyy.so
 0x00007feda99bb3b2 in CPyCppyy::CPPMethod::Call(CPyCppyy::CPPInstance*&, _object* const*, unsigned long, _object*, CPyCppyy::CallContext*) + 0x102 from /opt/root/lib/libcppyy.so
 0x00007feda99c16b1 in <unknown> from 

SegmentationViolation: segfault in C++; program state was reset

## Calculating events, efficiencies and purities

In [44]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)

#################### OBJECT TO FIND ################################
event_counts = {}
cuts_list = {}
efficiencies = {}
purities = {}

#################### EXTRACT EVENT COUNTS #######################
for hash in hash_available:
    obj = hist_file.Get(f"{hash}/ChannelIntegrals")
    obj_cut_list = hist_file.Get(f"{hash}/CutList")

    if obj:
      chann_integrals_str = obj.GetTitle()
      event_counts[hash] = json.loads(chann_integrals_str)

    if obj_cut_list:
      raw_title = obj_cut_list.GetTitle()
      cleaned_str = raw_title.strip("[] ").replace('"', '')
      cuts_list[hash] = [c.strip() for c in cleaned_str.split(',') if c.strip()]

hist_file.Close()
#################################################################

#################### CALCULATE EFFICIENCIES AND PURITIES #######################
for hash in hash_available:
  if hash not in event_counts:
     continue

  efficiencies[hash] = {}
  purities[hash] = {}

  print(event_counts[hash])

  mc_sum = int(event_counts[hash].get("MC sum", 0))

  for flag in channName.values():
      total_events = int(event_counts.get(nocut_hash, {}).get(flag, 0))
      passed_events = int(event_counts[hash].get(flag, 0))
      efficiencies[hash][flag] = passed_events / total_events if total_events > 0 else 0
      purities[hash][flag] = passed_events / mc_sum if mc_sum > 0 else 0
###############################################################################

################## PRINT RESULTS ##################
for hash in hash_available:
    if hash not in efficiencies:
        continue
    bkg = 0
    print(f"Results for cut list: {cuts_list.get(hash, [])}")
    for flag in channName.values():
        event_count = int(event_counts.get(hash, {}).get(flag, 0))
        mc_sum = int(event_counts.get(hash, {}).get("MC sum", 0))
        eff = efficiencies[hash].get(flag, 0)
        pur = purities[hash].get(flag, 0)

        if flag != "3pi0" and flag != "MC sum" and flag != "Data":
            bkg += event_count
        print(f"  Flag: {flag}, Event Count: {event_count / 10**6:.2f} x 10^6, Efficiency: {eff * 100:.2f}%, Purity: {pur * 100:.2f}%, (MC - Data) / Data: {100*(mc_sum - event_count) / event_count if event_count > 0 else 0:.2f}%")
    print(f"  Background (sum of non-signal, non-MC, non-Data flags): {bkg / 10**6:.2f} x 10^6")
    print()
    




{'Data': 48213558.0, 'Signal': 3660.2524782414293, 'Regeneration': 9177.777691115967, 'Omega': 51965.17903441543, '3pi0': 46277293.72674556, 'Semileptonic': 166773.3640758196, 'Other': 5529615.583313252, 'pi+pi-pi+pi-': 84.83279847592945, 'MC sum': 52038570.71613684}
{'Data': 39931794.0, 'Signal': 2903.543915836142, 'Regeneration': 7711.8669334519145, 'Omega': 21956.990453529997, '3pi0': 39573855.7597528, 'Semileptonic': 53518.18479851486, 'Other': 3561574.9670542367, 'pi+pi-pi+pi-': 84.83279847592945, 'MC sum': 43221606.145706736}
{'Data': 33956742.0, 'Signal': 2983.852298393359, 'Regeneration': 7075.055392892565, 'Omega': 1229.5100259111377, '3pi0': 37267656.90441694, 'Semileptonic': 1257.7876254031146, 'Other': 963818.2255004455, 'pi+pi-pi+pi-': 63.341822862027406, 'MC sum': 38244084.67708279}
{'Data': 33012014.0, 'Signal': 2883.1840442019225, 'Regeneration': 6765.132902460497, 'Omega': 93.88163031336188, '3pi0': 36221033.94987611, 'Semileptonic': 251.10508348875157, 'Other': 924128